# 06_graphgrued_ordered5_window_experiments

Train a **lighter, leakage-safe benchmark** on W1/W2/W4 windows. This keeps the original ordered-5 direction, but removes the old `is_completed` direction and explicitly tracks **speed vs. accuracy**.

In [4]:
%run utilities

StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 5, Finished, Available, Finished, True)

In [11]:
from pyspark.sql import functions as F

df = spark.table(INPUT_TABLE)  # hoặc tên bảng mà 03_v7 đang đọc

display(
    df.groupBy("obs_window_weeks", "split_name", "risk_label_5_idx")
      .count()
      .orderBy("obs_window_weeks", "split_name", "risk_label_5_idx")
)

StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 233718f6-7dcb-4ceb-9ab3-5f593abfd047)

In [12]:
display(
    df.groupBy("obs_window_weeks", "risk_label_5_idx")
      .count()
      .orderBy("obs_window_weeks", "risk_label_5_idx")
)

StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3f1be645-12b6-4d59-a7e1-4c1fd16d9543)

In [13]:
from pyspark.sql import functions as F

exclude_cols = {
    "user_id", "course_id", "enroll_time", "split_name", "imputer_name",
    "obs_window_weeks",
    "risk_label_5_idx", "risk_label_5_surrogate",
    "label_d_flag", "label_e_flag",
    "risk_label_5_name_a", "risk_label_5_name_b", "risk_label_5_name_c",
    "risk_label_5_name_d", "risk_label_5_name_e"
}

feature_cols = [c for c in df.columns if c not in exclude_cols]

zero_expr = None
for c in feature_cols:
    expr = (F.coalesce(F.col(c).cast("double"), F.lit(0.0)) != 0.0)
    zero_expr = expr if zero_expr is None else (zero_expr | expr)

display(
    df.groupBy("obs_window_weeks", "split_name", "imputer_name")
      .agg(
          F.count("*").alias("n_rows"),
          F.sum(F.when(~zero_expr, 1).otherwise(0)).alias("all_zero_rows")
      )
      .withColumn("all_zero_pct", F.round(F.col("all_zero_rows") / F.col("n_rows") * 100, 2))
      .orderBy("obs_window_weeks", "split_name", "imputer_name")
)

StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4530d89b-85aa-4d2f-bea9-f150584cb984)

In [14]:
key_cols = [c for c in [
    "video_count_week1", "video_count_week2", "video_count_week3", "video_count_week4",
    "problem_attempt_count_week1", "problem_attempt_count_week2", "problem_attempt_count_week3", "problem_attempt_count_week4",
    "rewind_count_week1", "rewind_count_week2", "rewind_count_week3", "rewind_count_week4",
    "chapter_coverage_ratio", "chapter_progress_ratio"
] if c in df.columns]

display(
    df.groupBy("obs_window_weeks", "split_name", "imputer_name")
      .agg(*[F.avg(F.coalesce(F.col(c).cast("double"), F.lit(0.0))).alias(c) for c in key_cols])
      .orderBy("obs_window_weeks", "split_name", "imputer_name")
)

StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 464dac0c-aaf6-4ac7-97b6-1360979bd060)

In [17]:
import pandas as pd
from collections import Counter

print("Unique y_true:", sorted(set(y_train)))
print("Unique y_pred:", sorted(set(y_valid)))

print("y_true dist:", Counter(y_train))
print("y_pred dist:", Counter(y_valid))

cm_df = pd.crosstab(
    pd.Series(y_train, name="true"),
    pd.Series(y_valid, name="pred"),
    dropna=False
)
print(cm_df)

StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 18, Finished, Available, Finished, False)

Unique y_true: [0, 1, 2, 3, 4]
Unique y_pred: [0, 1, 2, 3, 4]
y_true dist: Counter({3: 51500, 4: 14340, 1: 6525, 2: 6177, 0: 1785})
y_pred dist: Counter({3: 6438, 4: 1793, 1: 815, 2: 772, 0: 223})
pred    0    1    2     3     4
true                           
0       9   15   12   139    36
1      13   57   65   482   154
2      24   74   53   490   147
3     140  510  508  4168  1146
4      37  159  134  1159   310


In [18]:
result_rows = []
history_frames = []

for obs_w in WINDOWS_TO_RUN:
    for imputer_name in IMPUTERS_TO_RUN:
        pdf = pdf_all[(pdf_all['obs_window_weeks'] == obs_w) & (pdf_all['imputer_name'] == imputer_name)].copy()
        if pdf.empty:
            print('skip empty', obs_w, imputer_name)
            continue

        train_pd = pdf[pdf['split_name'] == 'train'].copy()
        valid_pd = pdf[pdf['split_name'] == 'valid'].copy()
        test_pd  = pdf[pdf['split_name'] == 'test'].copy()
        if train_pd.empty or valid_pd.empty or test_pd.empty:
            print('skip incomplete split', obs_w, imputer_name)
            continue

        for part in [train_pd, valid_pd, test_pd]:
            # remove leakage / legacy columns from features only
            for c in LEAKAGE_COLS:
                if c in part.columns and c != TARGET_COL:
                    part.drop(columns=[c], inplace=True)

        seq_train, static_train, y_train, seq_feats, static_feats = build_sequence_matrices_windowed(train_pd, TARGET_COL, ID_COLS, obs_w)
        seq_valid, static_valid, y_valid, _, _ = build_sequence_matrices_windowed(valid_pd, TARGET_COL, ID_COLS, obs_w)
        seq_test,  static_test,  y_test,  _, _ = build_sequence_matrices_windowed(test_pd,  TARGET_COL, ID_COLS, obs_w)
        print("obs_window_weeks:", obs_w)
        print("imputer_name:", imputer_name)

        print("train shape seq/static:", seq_train.shape, static_train.shape)
        print("valid shape seq/static:", seq_valid.shape, static_valid.shape)
        print("test  shape seq/static:", seq_test.shape, static_test.shape)

        print("train y unique:", sorted(pd.Series(y_train).dropna().unique().tolist()))
        print("valid y unique:", sorted(pd.Series(y_valid).dropna().unique().tolist()))
        print("test  y unique:", sorted(pd.Series(y_test).dropna().unique().tolist()))

StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 19, Finished, Available, Finished, False)

obs_window_weeks: 1
imputer_name: median
train shape seq/static: (80327, 1, 9) (80327, 52)
valid shape seq/static: (10041, 1, 9) (10041, 52)
test  shape seq/static: (10041, 1, 9) (10041, 52)
train y unique: [0, 1, 2, 3, 4]
valid y unique: [0, 1, 2, 3, 4]
test  y unique: [0, 1, 2, 3, 4]
obs_window_weeks: 1
imputer_name: knn
train shape seq/static: (80327, 1, 9) (80327, 52)
valid shape seq/static: (10041, 1, 9) (10041, 52)
test  shape seq/static: (10041, 1, 9) (10041, 52)
train y unique: [0, 1, 2, 3, 4]
valid y unique: [0, 1, 2, 3, 4]
test  y unique: [0, 1, 2, 3, 4]
obs_window_weeks: 1
imputer_name: graph_only
train shape seq/static: (80327, 1, 9) (80327, 52)
valid shape seq/static: (10041, 1, 9) (10041, 52)
test  shape seq/static: (10041, 1, 9) (10041, 52)
train y unique: [0, 1, 2, 3, 4]
valid y unique: [0, 1, 2, 3, 4]
test  y unique: [0, 1, 2, 3, 4]
obs_window_weeks: 1
imputer_name: graph_adv
train shape seq/static: (80327, 1, 9) (80327, 52)
valid shape seq/static: (10041, 1, 9) (10041

In [ ]:
from pyspark.sql import functions as F

exclude_cols = {
    "user_id", "course_id", "enroll_time", "split_name", "imputer_name",
    "obs_window_weeks",
    "risk_label_5_idx", "risk_label_5_surrogate",
    "label_d_flag", "label_e_flag",
    "risk_label_5_name_a", "risk_label_5_name_b", "risk_label_5_name_c",
    "risk_label_5_name_d", "risk_label_5_name_e"
}

feature_cols = [c for c in df.columns if c not in exclude_cols]

zero_expr = None
for c in feature_cols:
    expr = (F.coalesce(F.col(c).cast("double"), F.lit(0.0)) != 0.0)
    zero_expr = expr if zero_expr is None else (zero_expr | expr)

display(
    df.groupBy("obs_window_weeks", "split_name", "imputer_name")
      .agg(
          F.count("*").alias("n_rows"),
          F.sum(F.when(~zero_expr, 1).otherwise(0)).alias("all_zero_rows")
      )
      .withColumn("all_zero_pct", F.round(F.col("all_zero_rows") / F.col("n_rows") * 100, 2))
      .orderBy("obs_window_weeks", "split_name", "imputer_name")
)

In [5]:

INPUT_TABLE = "dbo.tbl_graphgrued_ordered5_windowed"
RESULT_TABLE = "dbo.tbl_graphgrued_ordered5_window_results"
HISTORY_TABLE = "dbo.tbl_graphgrued_ordered5_window_history"
TARGET_COL = "risk_label_5_idx"
ID_COLS = ["user_id", "course_id", "split_name", "imputer_name", "obs_window_weeks", "obs_window_days"]

WINDOWS_TO_RUN = [1, 2, 4]
IMPUTERS_TO_RUN = ["median", "knn", "graph_only", "graph_adv"]
MODELS_TO_RUN = [
    ("GRUAttentionLite", "gru"),
    ("BiLSTM", "bilstm"),
]
LOSS_SPECS = [
    {"loss_name": "cb_focal", "gamma": 2.0, "ordinal_lambda": 0.0},
    {"loss_name": "cb_focal_ordinal", "gamma": 2.0, "ordinal_lambda": 0.25},
]

HIDDEN_DIM = 96
LR = 3e-3
EPOCHS = 30
PATIENCE = 6
BATCH_SIZE = 256


StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 6, Finished, Available, Finished, False)

In [6]:

from pyspark.sql import functions as F
import pandas as pd, numpy as np, time

df_all = spark.table(INPUT_TABLE)
print("rows:", df_all.count())
display(df_all.limit(5))


StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 7, Finished, Available, Finished, False)

rows: 1807362


SynapseWidget(Synapse.DataFrame, 65ebcc5f-1617-497f-9cf2-d0d977f3e39a)

In [7]:

LEAKAGE_COLS = [
    'risk_label_5_idx',
    'label_d_flag',
    'label_e_flag',
    'is_completed',
    'completion_rate',
    'risk_label_5_surrogate',
    'risk_label_5_name_a',
    'risk_label_5_name_b',
    'risk_label_5_name_c',
    'risk_label_5_name_d',
    'risk_label_5_name_e',
    'course_norm_score',
    'course_raw_score_sum',
    'problems_passed',
    'total_problems',
]

SAFE_REQUIRED = ['imputer_name','split_name','obs_window_weeks',TARGET_COL]

pdf_all = df_all.toPandas()
missing = [c for c in SAFE_REQUIRED if c not in pdf_all.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

pdf_all[TARGET_COL] = pd.to_numeric(pdf_all[TARGET_COL], errors='coerce').astype('Int64')
pdf_all = pdf_all[pdf_all[TARGET_COL].notna()].copy()
pdf_all[TARGET_COL] = pdf_all[TARGET_COL].astype(int)


StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 8, Finished, Available, Finished, False)

In [21]:
import re
import numpy as np
import pandas as pd

def build_sequence_matrices_windowed_fixed(
    df: pd.DataFrame,
    label_col: str,
    id_cols,
    obs_w: int,
):
    """
    Build sequence/static matrices strictly using only information up to obs_w weeks.

    Rules:
    - sequence cols: only *_week1 ... *_week{obs_w}
    - static cols:
        + keep non-week/non-window-suffixed cols
        + keep *_w{k} only if k <= obs_w
        + drop *_w{k} if k > obs_w
    """
    week_map = {}
    static_cols = []

    for c in df.columns:
        if c in id_cols or c == label_col:
            continue

        m_week = re.search(r"^(.*)_week(\d+)$", c)
        if m_week:
            base = m_week.group(1)
            wk = int(m_week.group(2))
            if 1 <= wk <= obs_w:
                week_map.setdefault(base, {})[wk] = c
            continue

        m_win = re.search(r"^(.*)_w(\d+)$", c)
        if m_win:
            k = int(m_win.group(2))
            if k <= obs_w:
                static_cols.append(c)
            continue

        # non-week / non-window feature => keep
        static_cols.append(c)

    seq_bases = sorted([b for b, mp in week_map.items() if all(k in mp for k in range(1, obs_w + 1))])

    n = len(df)
    seq_x = np.zeros((n, obs_w, len(seq_bases)), dtype=np.float32)

    for j, b in enumerate(seq_bases):
        for w in range(1, obs_w + 1):
            col = week_map[b][w]
            seq_x[:, w - 1, j] = (
                pd.to_numeric(df[col], errors="coerce")
                .fillna(0.0)
                .astype(np.float32)
                .values
            )

    static_x = np.zeros((n, len(static_cols)), dtype=np.float32)
    for j, c in enumerate(static_cols):
        static_x[:, j] = (
            pd.to_numeric(df[c], errors="coerce")
            .fillna(0.0)
            .astype(np.float32)
            .values
        )

    y = df[label_col].astype(int).values
    return seq_x, static_x, y, seq_bases, static_cols

StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 27, Finished, Available, Finished, False)

In [22]:

result_rows = []
history_frames = []

for obs_w in WINDOWS_TO_RUN:
    for imputer_name in IMPUTERS_TO_RUN:
        pdf = pdf_all[(pdf_all['obs_window_weeks'] == obs_w) & (pdf_all['imputer_name'] == imputer_name)].copy()
        if pdf.empty:
            print('skip empty', obs_w, imputer_name)
            continue

        train_pd = pdf[pdf['split_name'] == 'train'].copy()
        valid_pd = pdf[pdf['split_name'] == 'valid'].copy()
        test_pd  = pdf[pdf['split_name'] == 'test'].copy()
        if train_pd.empty or valid_pd.empty or test_pd.empty:
            print('skip incomplete split', obs_w, imputer_name)
            continue

        for part in [train_pd, valid_pd, test_pd]:
            # remove leakage / legacy columns from features only
            for c in LEAKAGE_COLS:
                if c in part.columns and c != TARGET_COL:
                    part.drop(columns=[c], inplace=True)

        seq_train, static_train, y_train, seq_feats, static_feats = build_sequence_matrices_windowed_fixed(train_pd, TARGET_COL, ID_COLS, obs_w)
        seq_valid, static_valid, y_valid, _, _ = build_sequence_matrices_windowed_fixed(valid_pd, TARGET_COL, ID_COLS, obs_w)
        seq_test,  static_test,  y_test,  _, _ = build_sequence_matrices_windowed_fixed(test_pd,  TARGET_COL, ID_COLS, obs_w)

        assert seq_train.shape[1] == obs_w, f"seq_train has {seq_train.shape[1]} steps, expected {obs_w}"
        assert seq_valid.shape[1] == obs_w, f"seq_valid has {seq_valid.shape[1]} steps, expected {obs_w}"
        assert seq_test.shape[1] == obs_w,  f"seq_test has {seq_test.shape[1]} steps, expected {obs_w}"

        future_static = []
        for c in static_feats:
            m = re.search(r"_w(\d+)$", c)
            if m and int(m.group(1)) > obs_w:
                future_static.append(c)

        assert len(future_static) == 0, f"Future-window static cols leaked into window={obs_w}: {future_static[:20]}"

        train_ds = SequenceDataset(seq_train, static_train, y_train)
        valid_ds = SequenceDataset(seq_valid, static_valid, y_valid)
        test_ds  = SequenceDataset(seq_test, static_test, y_test)

        for display_name, model_key in MODELS_TO_RUN:
            for loss_spec in LOSS_SPECS:
                print(f"run window={obs_w} imputer={imputer_name} model={display_name} loss={loss_spec['loss_name']}")
                t0 = time.time()
                fit_res = fit_sequence_model_plus(
                    train_ds=train_ds,
                    valid_ds=valid_ds,
                    seq_input_dim=max(1, seq_train.shape[2]),
                    static_input_dim=static_train.shape[1],
                    num_classes=5,
                    model_type=model_key,
                    hidden_dim=HIDDEN_DIM,
                    lr=LR,
                    epochs=EPOCHS,
                    patience=PATIENCE,
                    batch_size=BATCH_SIZE,
                    loss_name=loss_spec['loss_name'],
                    gamma=loss_spec['gamma'],
                    ordinal_lambda=loss_spec['ordinal_lambda'],
                    use_one_cycle=True,
                )
                runtime_sec = float(time.time() - t0)

                valid_metrics = evaluate_sequence_model_plus(fit_res.model, valid_ds, num_classes=5)
                test_metrics  = evaluate_sequence_model_plus(fit_res.model, test_ds,  num_classes=5)

                result_rows.append({
                    'obs_window_weeks': int(obs_w),
                    'obs_window_days': int(obs_w * 7),
                    'imputer_name': imputer_name,
                    'model_name': display_name,
                    'loss_name': loss_spec['loss_name'],
                    'target_col': TARGET_COL,
                    'best_threshold': fit_res.best_threshold,
                    'monitor_metric': fit_res.best_metric_name,
                    'valid_accuracy': valid_metrics['accuracy'],
                    'valid_macro_f1': valid_metrics['macro_f1'],
                    'valid_weighted_f1': valid_metrics['weighted_f1'],
                    'valid_label_D_f1': valid_metrics['label_D_f1'],
                    'valid_auc_ovr': valid_metrics['auc_ovr'],
                    'test_accuracy': test_metrics['accuracy'],
                    'test_macro_f1': test_metrics['macro_f1'],
                    'test_weighted_f1': test_metrics['weighted_f1'],
                    'test_label_D_f1': test_metrics['label_D_f1'],
                    'test_label_D_precision': test_metrics['label_D_precision'],
                    'test_label_D_recall': test_metrics['label_D_recall'],
                    'test_label_E_f1': test_metrics['label_E_f1'],
                    'test_auc_ovr': test_metrics['auc_ovr'],
                    'n_seq_features': int(seq_train.shape[2]),
                    'n_static_features': int(static_train.shape[1]),
                    'runtime_sec': runtime_sec,
                })

                hist = fit_res.history.copy()
                hist['obs_window_weeks'] = int(obs_w)
                hist['imputer_name'] = imputer_name
                hist['model_name'] = display_name
                hist['loss_name'] = loss_spec['loss_name']
                hist['target_col'] = TARGET_COL
                hist['runtime_sec_total'] = runtime_sec
                history_frames.append(hist)


StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 28, Finished, Available, Finished, False)

run window=1 imputer=median model=GRUAttentionLite loss=cb_focal
run window=1 imputer=median model=GRUAttentionLite loss=cb_focal_ordinal
run window=1 imputer=median model=BiLSTM loss=cb_focal
run window=1 imputer=median model=BiLSTM loss=cb_focal_ordinal
run window=1 imputer=knn model=GRUAttentionLite loss=cb_focal
run window=1 imputer=knn model=GRUAttentionLite loss=cb_focal_ordinal
run window=1 imputer=knn model=BiLSTM loss=cb_focal
run window=1 imputer=knn model=BiLSTM loss=cb_focal_ordinal
run window=1 imputer=graph_only model=GRUAttentionLite loss=cb_focal
run window=1 imputer=graph_only model=GRUAttentionLite loss=cb_focal_ordinal
run window=1 imputer=graph_only model=BiLSTM loss=cb_focal
run window=1 imputer=graph_only model=BiLSTM loss=cb_focal_ordinal
run window=1 imputer=graph_adv model=GRUAttentionLite loss=cb_focal
run window=1 imputer=graph_adv model=GRUAttentionLite loss=cb_focal_ordinal
run window=1 imputer=graph_adv model=BiLSTM loss=cb_focal
run window=1 imputer=graph

In [10]:

result_pd = pd.DataFrame(result_rows)
if result_pd.empty:
    raise ValueError('No result rows were produced.')

result_pd = result_pd.sort_values(
    ['obs_window_weeks', 'test_macro_f1', 'test_label_D_f1', 'runtime_sec'],
    ascending=[True, False, False, True]
)

spark.sql(f"DROP TABLE IF EXISTS {RESULT_TABLE}")
spark.createDataFrame(result_pd).write.mode('overwrite').format('delta').saveAsTable(RESULT_TABLE)
print('saved:', RESULT_TABLE)
display(spark.table(RESULT_TABLE))

if history_frames:
    hist_pd = pd.concat(history_frames, ignore_index=True)
    spark.sql(f"DROP TABLE IF EXISTS {HISTORY_TABLE}")
    spark.createDataFrame(hist_pd).write.mode('overwrite').format('delta').saveAsTable(HISTORY_TABLE)
    print('saved:', HISTORY_TABLE)


StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 11, Finished, Available, Finished, False)

saved: dbo.tbl_graphgrued_ordered5_window_results


SynapseWidget(Synapse.DataFrame, c245f722-9157-4674-9469-631611455c2f)

saved: dbo.tbl_graphgrued_ordered5_window_history


In [19]:
# ============================================
# DEBUG ONE RUN ONLY
# Ordered-5 / window benchmark
# ============================================

# Nếu bạn dùng notebook utilities riêng thì %run lại cho chắc
# %run utilities_graphgrued_ordered5_fixed

import time
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# =========================
# 0) CONFIG
# =========================
WINDOW_TABLE = "dbo.tbl_graphgrued_ordered5_windowed"   # sửa nếu bảng bạn khác tên
TARGET_COL = "risk_label_5_idx"

DEBUG_WINDOW = 1
DEBUG_IMPUTER = "median"          # median | knn | graph_only | graph_adv
DEBUG_MODEL = "BiLSTM"            # GRUAttentionLite | BiLSTM
DEBUG_LOSS = "cb_focal"           # cb_focal | cb_focal_ordinal

SEED = 42
HIDDEN_DIM = 64
LR = 1e-3
EPOCHS = 20
PATIENCE = 5
BATCH_SIZE = 256
USE_ONE_CYCLE = True

ID_COLS = ["user_id", "course_id", "enroll_time"]
DROP_ALWAYS = {
    "split_name",
    "imputer_name",
    "obs_window_weeks",
    "risk_label_5_surrogate",
    "label_d_flag",
    "label_e_flag",
    "risk_label_5_name_a",
    "risk_label_5_name_b",
    "risk_label_5_name_c",
    "risk_label_5_name_d",
    "risk_label_5_name_e",
    "is_completed",
    "completion_rate",
    "course_raw_score_sum",
    "course_norm_score",
    "problems_passed",
    "total_problems",
}

# =========================
# 1) LOAD ONE RUN DATA
# =========================
base_df = spark.table(WINDOW_TABLE)

run_sdf = (
    base_df
    .filter(
        (F.col("obs_window_weeks") == DEBUG_WINDOW) &
        (F.col("imputer_name") == DEBUG_IMPUTER)
    )
)

print("Rows in selected run:", run_sdf.count())
print("Selected config:")
print({
    "window": DEBUG_WINDOW,
    "imputer": DEBUG_IMPUTER,
    "model": DEBUG_MODEL,
    "loss": DEBUG_LOSS,
})

display(
    run_sdf.groupBy("split_name", TARGET_COL)
          .count()
          .orderBy("split_name", TARGET_COL)
)

run_df = run_sdf.toPandas().copy()

# =========================
# 2) BASIC SANITY CHECKS
# =========================
print("\n=== Basic checks ===")
print("Columns:", len(run_df.columns))
print("Unique target overall:", sorted(pd.Series(run_df[TARGET_COL]).dropna().unique().tolist()))
print("Target dtype:", run_df[TARGET_COL].dtype)

for split_name in ["train", "valid", "test"]:
    tmp = run_df[run_df["split_name"] == split_name].copy()
    print(f"{split_name} rows:", len(tmp))
    print(f"{split_name} target unique:", sorted(pd.Series(tmp[TARGET_COL]).dropna().unique().tolist()))

# =========================
# 3) SPLIT DATAFRAMES
# =========================
train_df = run_df[run_df["split_name"] == "train"].reset_index(drop=True).copy()
valid_df = run_df[run_df["split_name"] == "valid"].reset_index(drop=True).copy()
test_df  = run_df[run_df["split_name"] == "test"].reset_index(drop=True).copy()

assert len(train_df) > 0 and len(valid_df) > 0 and len(test_df) > 0, "Train/valid/test bị rỗng"
assert sorted(pd.Series(train_df[TARGET_COL]).dropna().unique().tolist()) == [0, 1, 2, 3, 4], "Train target sai"
assert sorted(pd.Series(valid_df[TARGET_COL]).dropna().unique().tolist()) == [0, 1, 2, 3, 4], "Valid target sai"
assert sorted(pd.Series(test_df[TARGET_COL]).dropna().unique().tolist()) == [0, 1, 2, 3, 4], "Test target sai"

# =========================
# 4) DROP LEAKAGE COLUMNS
# =========================
drop_cols = set(ID_COLS) | DROP_ALWAYS | {TARGET_COL}

feature_cols = [c for c in train_df.columns if c not in drop_cols]

print("\n=== Feature checks ===")
print("n feature cols:", len(feature_cols))
print("first 30 feature cols:", feature_cols[:30])

# check xem có cột leakage còn lọt không
leak_like = [
    c for c in feature_cols
    if ("risk_label" in c.lower()) or
       ("label_" in c.lower()) or
       (c.lower() in {"is_completed", "completion_rate"})
]
print("leak-like feature cols:", leak_like)

assert len(leak_like) == 0, f"Còn leakage cols trong feature: {leak_like}"

train_work = train_df[ID_COLS + [TARGET_COL] + feature_cols].copy()
valid_work = valid_df[ID_COLS + [TARGET_COL] + feature_cols].copy()
test_work  = test_df[ID_COLS + [TARGET_COL] + feature_cols].copy()

# =========================
# 5) BUILD MATRICES
# =========================
seq_train, static_train, y_train, seq_bases, static_cols = build_sequence_matrices(
    train_work, label_col=TARGET_COL, id_cols=ID_COLS
)
seq_valid, static_valid, y_valid, _, _ = build_sequence_matrices(
    valid_work, label_col=TARGET_COL, id_cols=ID_COLS
)
seq_test, static_test, y_test, _, _ = build_sequence_matrices(
    test_work, label_col=TARGET_COL, id_cols=ID_COLS
)

print("\n=== Matrix shapes ===")
print("seq_train:", seq_train.shape, "static_train:", static_train.shape)
print("seq_valid:", seq_valid.shape, "static_valid:", static_valid.shape)
print("seq_test :", seq_test.shape,  "static_test :", static_test.shape)

print("n seq bases:", len(seq_bases))
print("seq bases:", seq_bases)
print("n static cols:", len(static_cols))
print("first 30 static cols:", static_cols[:30])

print("\n=== y distributions ===")
print("train y:", Counter(y_train.tolist()))
print("valid y:", Counter(y_valid.tolist()))
print("test  y:", Counter(y_test.tolist()))

# =========================
# 6) ALL-ZERO CHECK
# =========================
def pct_all_zero_seq(x):
    row_zero = np.all(np.isclose(x, 0.0), axis=(1, 2))
    return row_zero.mean() * 100.0

def pct_all_zero_static(x):
    row_zero = np.all(np.isclose(x, 0.0), axis=1)
    return row_zero.mean() * 100.0

print("\n=== Zero checks ===")
print("train all-zero seq %   :", round(pct_all_zero_seq(seq_train), 4))
print("valid all-zero seq %   :", round(pct_all_zero_seq(seq_valid), 4))
print("test  all-zero seq %   :", round(pct_all_zero_seq(seq_test), 4))
print("train all-zero static %:", round(pct_all_zero_static(static_train), 4))
print("valid all-zero static %:", round(pct_all_zero_static(static_valid), 4))
print("test  all-zero static %:", round(pct_all_zero_static(static_test), 4))

# =========================
# 7) DATASET
# =========================
train_ds = SequenceDataset(seq_train, static_train, y_train)
valid_ds = SequenceDataset(seq_valid, static_valid, y_valid)
test_ds  = SequenceDataset(seq_test, static_test, y_test)

# =========================
# 8) TRAIN ONE MODEL
# =========================
print("\n=== Training one run ===")
start_time = time.time()

fit_res = fit_sequence_model_plus(
    train_ds=train_ds,
    valid_ds=valid_ds,
    seq_input_dim=seq_train.shape[2],
    static_input_dim=static_train.shape[1],
    num_classes=5,
    model_type=DEBUG_MODEL,
    hidden_dim=HIDDEN_DIM,
    lr=LR,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    loss_name=DEBUG_LOSS,
    gamma=2.0,
    ordinal_lambda=0.2,
    use_one_cycle=USE_ONE_CYCLE,
)

runtime_sec = time.time() - start_time
print("runtime_sec:", round(runtime_sec, 2))
print("best_threshold:", getattr(fit_res, "best_threshold", None))
print("best_metric_name:", getattr(fit_res, "best_metric_name", None))
print("best_metric_value:", getattr(fit_res, "best_metric_value", None))

if hasattr(fit_res, "history") and fit_res.history is not None:
    print("\n=== Fit history tail ===")
    display(fit_res.history.tail(10))

# =========================
# 9) EVALUATE
# =========================
valid_metrics = evaluate_sequence_model_plus(fit_res.model, valid_ds, num_classes=5)
test_metrics  = evaluate_sequence_model_plus(fit_res.model, test_ds, num_classes=5)

print("\n=== Valid metrics ===")
for k, v in valid_metrics.items():
    print(k, ":", v)

print("\n=== Test metrics ===")
for k, v in test_metrics.items():
    print(k, ":", v)

# =========================
# 10) GET RAW PREDICTIONS
# =========================
import torch
from torch.utils.data import DataLoader

def predict_with_probs(model, ds, num_classes=5, batch_size=512, device=None):
    dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    model.eval()
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for seq_x, static_x, y in loader:
            seq_x = seq_x.to(dev)
            static_x = static_x.to(dev)

            logits = model(seq_x, static_x)
            prob = torch.softmax(logits, dim=1)
            pred = prob.argmax(dim=1)

            y_true.extend(y.numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())
            y_prob.append(prob.cpu().numpy())

    y_prob = np.vstack(y_prob)
    return np.array(y_true), np.array(y_pred), y_prob

y_true, y_pred, y_prob = predict_with_probs(fit_res.model, test_ds, num_classes=5)

# =========================
# 11) DEBUG PREDICTIONS
# =========================
print("\n=== Prediction distribution ===")
print("y_true unique:", sorted(set(y_true.tolist())))
print("y_pred unique:", sorted(set(y_pred.tolist())))

print("y_true dist:", Counter(y_true.tolist()))
print("y_pred dist:", Counter(y_pred.tolist()))

cm = pd.crosstab(
    pd.Series(y_true, name="true"),
    pd.Series(y_pred, name="pred"),
    dropna=False
)
print("\n=== Confusion matrix ===")
print(cm)

print("\n=== First 10 probability rows ===")
print(np.round(y_prob[:10], 4))

print("\n=== Mean probability by class ===")
print(np.round(y_prob.mean(axis=0), 6))

print("\n=== Mean predicted class index ===")
print(float(np.mean(y_pred)))

# =========================
# 12) CLASSIFICATION REPORT
# =========================
print("\n=== Classification report ===")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0,1,2,3,4],
        target_names=["A", "B", "C", "D", "E"],
        zero_division=0
    )
)

# =========================
# 13) SAVE DEBUG OUTPUT
# =========================
debug_pred_df = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "prob_A": y_prob[:, 0],
    "prob_B": y_prob[:, 1],
    "prob_C": y_prob[:, 2],
    "prob_D": y_prob[:, 3],
    "prob_E": y_prob[:, 4],
})

print("\n=== Debug prediction sample ===")
display(debug_pred_df.head(20))

# nếu muốn ghi ra Spark table thì bật đoạn này
SAVE_DEBUG_TABLE = False
DEBUG_TABLE_NAME = "dbo.tbl_graphgrued_debug_one_run_preds"

if SAVE_DEBUG_TABLE:
    spark.createDataFrame(debug_pred_df).write.mode("overwrite").format("delta").saveAsTable(DEBUG_TABLE_NAME)
    print("Saved debug predictions to:", DEBUG_TABLE_NAME)

StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 21, Finished, Available, Finished, False)

Rows in selected run: 100409
Selected config:
{'window': 1, 'imputer': 'median', 'model': 'BiLSTM', 'loss': 'cb_focal'}


SynapseWidget(Synapse.DataFrame, b304a08d-a7ce-472d-bf33-f5d92dab59d1)


=== Basic checks ===
Columns: 96
Unique target overall: [0, 1, 2, 3, 4]
Target dtype: int64
train rows: 80327
train target unique: [0, 1, 2, 3, 4]
valid rows: 10041
valid target unique: [0, 1, 2, 3, 4]
test rows: 10041
test target unique: [0, 1, 2, 3, 4]

=== Feature checks ===
n feature cols: 87
first 30 feature cols: ['completed_questions_week1', 'correct_answers_week1', 'problem_attempt_count_week1', 'rewind_count_week1', 'rewind_distance_week1', 'total_score_week1', 'video_avg_chapter_week1', 'video_count_week1', 'video_watch_time_week1', 'obs_window_days', 'video_count_total_w1', 'video_watch_time_total_w1', 'rewind_count_total_w1', 'rewind_distance_total_w1', 'problem_attempt_count_total_w1', 'total_score_total_w1', 'correct_answers_total_w1', 'completed_questions_total_w1', 'avg_chapter_progress_w1', 'course_total_videos', 'chapter_coverage_ratio_w1', 'course_max_chapter', 'chapter_progress_ratio_w1', 'activity_signal_w1', 'is_active_w1', 'year_of_birth', 'num_courses_enrolled'

SynapseWidget(Synapse.DataFrame, fdf7054b-3575-44a2-98bf-a40cea27305f)


=== Valid metrics ===
accuracy : 0.5678717259237128
macro_f1 : 0.49527896605951743
weighted_f1 : 0.5991242115490284
macro_precision : 0.4888507832415495
macro_recall : 0.6601246329115369
auc_ovr : 0.8757471587821121
label_A_f1 : 0.22769953051643194
label_A_precision : 0.13099257258609048
label_A_recall : 0.8699551569506726
label_B_f1 : 0.5983471074380166
label_B_precision : 0.543
label_B_recall : 0.6662576687116565
label_C_f1 : 0.4334140435835351
label_C_precision : 0.4068181818181818
label_C_recall : 0.4637305699481865
label_D_f1 : 0.637420398261397
label_D_precision : 0.9125904486251809
label_D_recall : 0.4897483690587139
label_E_f1 : 0.5795137504982064
label_E_precision : 0.45085271317829456
label_E_recall : 0.8109313998884551

=== Test metrics ===
accuracy : 0.5645852006772234
macro_f1 : 0.49936213050051065
weighted_f1 : 0.5938877664911556
macro_precision : 0.4916264428606943
macro_recall : 0.6703083283118423
auc_ovr : 0.8751975690277627
label_A_f1 : 0.2252199413489736
label_A_pre

SynapseWidget(Synapse.DataFrame, 96b2f807-4d7c-4da1-b8a9-ca09289f7504)

In [25]:
# ============================================
# 14) VALIDATE + SAVE RESULTS SAFELY
# ============================================

result_pd = pd.DataFrame(result_rows)
if result_pd.empty:
    raise ValueError("No result rows were produced. Rerun cell 13 first.")

# --------
# sanity checks
# --------
print("n result rows:", len(result_pd))
print("columns:", result_pd.columns.tolist())

print("\n=== Distinct configs ===")
print("windows :", sorted(result_pd["obs_window_weeks"].unique().tolist()))
print("imputers:", sorted(result_pd["imputer_name"].unique().tolist()))
print("models  :", sorted(result_pd["model_name"].unique().tolist()))
print("losses  :", sorted(result_pd["loss_name"].unique().tolist()))

print("\n=== Quick metric summary ===")
print(result_pd[[
    "obs_window_weeks",
    "imputer_name",
    "model_name",
    "loss_name",
    "test_accuracy",
    "test_macro_f1",
    "test_label_D_f1",
    "test_auc_ovr",
    "runtime_sec"
]].head(20))

# check collapse / suspicious patterns
metric_cols = ["test_accuracy", "test_macro_f1", "test_label_D_f1", "test_auc_ovr"]
n_unique_metrics = {
    c: result_pd[c].nunique(dropna=False)
    for c in metric_cols if c in result_pd.columns
}
print("\nUnique metric counts:", n_unique_metrics)

# nếu tất cả metric đều chỉ có 1 giá trị lặp lại, cảnh báo
if all(v <= 1 for v in n_unique_metrics.values()):
    raise ValueError(
        f"All key metrics look constant across runs: {n_unique_metrics}. "
        "Cell 13 likely produced collapsed/invalid results. Do NOT save."
    )

# sort đẹp hơn
sort_cols = [c for c in [
    "obs_window_weeks",
    "test_macro_f1",
    "test_label_D_f1",
    "test_auc_ovr",
    "runtime_sec"
] if c in result_pd.columns]

ascending = []
for c in sort_cols:
    if c in ["obs_window_weeks", "runtime_sec"]:
        ascending.append(True)
    else:
        ascending.append(False)

result_pd = result_pd.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)

print("\n=== Top results preview ===")
display(result_pd.head(20))

# --------
# save to a NEW table first
# --------
RESULT_TABLE_SAFE = RESULT_TABLE + "_safe"
HISTORY_TABLE_SAFE = HISTORY_TABLE + "_safe"

spark.sql(f"DROP TABLE IF EXISTS {RESULT_TABLE_SAFE}")
spark.createDataFrame(result_pd).write.mode("overwrite").format("delta").saveAsTable(RESULT_TABLE_SAFE)
print("saved results to:", RESULT_TABLE_SAFE)

if history_frames:
    hist_pd = pd.concat(history_frames, ignore_index=True)

    # nhẹ nhàng check history
    print("\n=== History preview ===")
    print(hist_pd.head())

    spark.sql(f"DROP TABLE IF EXISTS {HISTORY_TABLE_SAFE}")
    spark.createDataFrame(hist_pd).write.mode("overwrite").format("delta").saveAsTable(HISTORY_TABLE_SAFE)
    print("saved history to:", HISTORY_TABLE_SAFE)

StatementMeta(, 9007f138-bb45-491b-880a-a0b9eee02f58, 57, Finished, Available, Finished, False)

n result rows: 48
columns: ['obs_window_weeks', 'obs_window_days', 'imputer_name', 'model_name', 'loss_name', 'target_col', 'best_threshold', 'monitor_metric', 'best_metric_value', 'valid_accuracy', 'valid_macro_f1', 'valid_weighted_f1', 'valid_label_D_f1', 'valid_auc_ovr', 'test_accuracy', 'test_macro_f1', 'test_weighted_f1', 'test_label_D_f1', 'test_label_D_precision', 'test_label_D_recall', 'test_label_E_f1', 'test_auc_ovr', 'n_seq_features', 'n_static_features', 'runtime_sec']

=== Distinct configs ===
windows : [1, 2, 4]
imputers: ['graph_adv', 'graph_only', 'knn', 'median']
models  : ['BiLSTM', 'GRUAttentionLite']
losses  : ['cb_focal', 'cb_focal_ordinal']

=== Quick metric summary ===
    obs_window_weeks imputer_name        model_name         loss_name  \
0                  1       median  GRUAttentionLite          cb_focal   
1                  1       median  GRUAttentionLite  cb_focal_ordinal   
2                  1       median            BiLSTM          cb_focal   
3      

SynapseWidget(Synapse.DataFrame, 1d0a2b49-b676-404d-9af4-33da9e10b19f)

saved results to: dbo.tbl_graphgrued_ordered5_window_results_safe

=== History preview ===
   epoch  train_loss  valid_accuracy  valid_macro_f1  valid_weighted_f1  \
0      1    1.029678        0.022209        0.008691           0.000965   
1      2    0.924810        0.228862        0.252336           0.236652   
2      3    0.717328        0.318395        0.277489           0.307915   
3      4    0.646959        0.385918        0.342545           0.432528   
4      5    0.610308        0.379245        0.342779           0.398039   

   valid_label_D_f1  valid_auc_ovr  obs_window_weeks imputer_name  \
0          0.000000       0.720667                 1       median   
1          0.189584       0.749540                 1       median   
2          0.290831       0.776639                 1       median   
3          0.468480       0.793580                 1       median   
4          0.396283       0.805978                 1       median   

         model_name loss_name        target